In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

df=pd.read_csv('dataset_for_dano_fuel.csv')

df['week'] = pd.to_datetime(df['week']).dt.tz_convert(None)
df = df.sort_values('week')

df['week_start'] = df['week'].dt.normalize() + pd.to_timedelta(1, 'D')
df['week_end']   = df['week_start'] + pd.to_timedelta(6, 'D')
df['week']   = (df['week'] - df['week'].min()).dt.days // 7 + 1 + 13  # приводим к номерам недель года

# bad_parties = df[((df['week'] == 14) | (df['week'] >= 19)) & (df['service_fee_amt'] == 69)]['party_rk']
# df = df[~((df['service_fee_amt'] == 69) & df['week'].isin([14] + list(range(19, 23))))]

In [ ]:
count_69 = df[
    (df['service_fee_amt'] == 69) &
    ((df['week'] == 14) | (df['week'] >= 19))
].shape[0]

print(count_69)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fee_order = sorted(df['service_fee_amt'].unique())
df_plot = df.pivot_table(index='week', columns='service_fee_amt', aggfunc='size', fill_value=0)
df_plot = df_plot[fee_order]

tbank_colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00', 
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]

fig, ax = plt.subplots(figsize=(12, 6))
df_plot.plot(kind='bar', stacked=True, color=tbank_colors, ax=ax)

# plt.subplots_adjust(right=0.5)
plt.xlabel('Номер недели')
plt.ylabel('Количество записей')
plt.title('Распределение сервисного сбора по неделям')
plt.legend(title='Сервисный Сбор', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

week_info = { '2025-03-30 21:00:00+00:00': { 'week_start': '2025-03-31', 'week_end': '2025-04-06', 'week_num': 14 }, '2025-04-06 21:00:00+00:00': { 'week_start': '2025-04-07', 'week_end': '2025-04-13', 'week_num': 15 }, '2025-04-13 21:00:00+00:00': { 'week_start': '2025-04-14', 'week_end': '2025-04-20', 'week_num': 16 }, '2025-04-20 21:00:00+00:00': { 'week_start': '2025-04-21', 'week_end': '2025-04-27', 'week_num': 17 }, '2025-04-27 21:00:00+00:00': { 'week_start': '2025-04-28', 'week_end': '2025-05-04', 'week_num': 18 }, '2025-05-04 21:00:00+00:00': { 'week_start': '2025-05-05', 'week_end': '2025-05-11', 'week_num': 19 }, '2025-05-11 21:00:00+00:00': { 'week_start': '2025-05-12', 'week_end': '2025-05-18', 'week_num': 20 }, '2025-05-18 21:00:00+00:00': { 'week_start': '2025-05-19', 'week_end': '2025-05-25', 'week_num': 21 }, '2025-05-25 21:00:00+00:00': { 'week_start': '2025-05-26', 'week_end': '2025-06-01', 'week_num': 22 } }

# Данные для таблицы
week_data = []
week_keys = sorted(week_info.keys())
for date in week_keys:
    info = week_info[date]
    week_data.append([date, info['week_num'], info['week_start'], info['week_end']])

week_table = pd.DataFrame(week_data, columns=['Raw Week', 'Week', 'Start', 'End'])

# Создаём фигуру только для таблицы
fig, ax = plt.subplots(figsize=(6, 3))
ax.axis('off')  # убираем оси
ax.set_facecolor('white')  # белый фон

# Указываем ширины колонок (в долях от фигуры)
col_widths = [0.4, 0.2, 0.2, 0.2]  # первая колонка шире

# Рисуем таблицу
table = ax.table(
    cellText=week_table.values,
    colLabels=week_table.columns,
    cellLoc='center',
    colLoc='center',
    colWidths=col_widths,
    loc='center'
)

# Красим только заголовки
tbank_colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00', 
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]

for (i, j), cell in table.get_celld().items():
    if i == 0:  # заголовки
        cell.set_facecolor(tbank_colors[0])  # ярко-жёлтый
        cell.set_text_props(weight='bold', color='black')

table.auto_set_font_size(False)
table.set_fontsize(10)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Проверим данные
total_rows = len(df)
empty_rows = df['region'].isna().sum() + (df['region'] == '').sum()

# Создаём DataFrame для графика
df_empty_check = pd.DataFrame({
    'Status': ['Empty', 'Non-Empty'],
    'Count': [empty_rows, total_rows - empty_rows]
})

# Красивый барплот
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(df_empty_check['Status'], df_empty_check['Count'], color=['#ffd600', '#a0a0a0'])

# Подписи на барах
for bar in bars:
    height = bar.get_height()
    # ax.text(bar.get_x() + bar.get_width()/2, height + total_rows*0.01, f'{int(height)}', 
    #         ha='center', va='bottom', fontweight='bold')
    ax.text(bar.get_x() + bar.get_width()/2, total_rows*0.01, f'{int(height)}', 
            ha='center', va='bottom', fontweight='bold')

# Настройка графика
ax.set_ylabel('Количество записей')
ax.set_title('Проверка поля region')
ax.set_ylim(0, total_rows*1.05)

plt.show()

In [ ]:
plt.figure(figsize=(14,6))

# boxplot без точек
sns.boxplot(
    data=df,
    x='week',
    y='service_fee_amt',
    showfliers=False
)

# добавляем только service_fee_amt = 69
df_69 = df[df['service_fee_amt'] == 69]

sns.stripplot(
    data=df_69,
    x='week',
    y='service_fee_amt',
    color='red',
    size=7,
    jitter=True,
    label='69 — выбросы'
)

plt.title("Boxplot + красные точки выбросов (69)")
plt.xlabel("Неделя")
plt.ylabel("service_fee_amt")
plt.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

tbank_colors = [
    '#ffd600',  # яркий фирменный жёлтый
    '#f3c400',  
    '#d4a700',  
    '#b08500',  
    '#8c6a00',  
    '#a0a0a0',  
    '#787878',  
    '#505a65',  
    '#2f3a45',  
    '#1d1d1d',  
    '#000000'   
]

# ====== Данные ======
df_14 = df[df['week'] == 14]['service_fee_amt'].value_counts().sort_index()
df_19_22 = df[df['week'].between(19, 22)]['service_fee_amt'].value_counts().sort_index()

# Функция для explode и цветов
def get_explode_colors(values):
    explode = [0.1 if v == 69 else 0 for v in values.index]
    colors = ['red' if v == 69 else tbank_colors[i % len(tbank_colors)] 
              for i, v in enumerate(values.index)]
    return explode, colors

explode_14, colors_14 = get_explode_colors(df_14)
explode_19_22, colors_19_22 = get_explode_colors(df_19_22)

# ====== Рисуем фигуру ======
fig, axes = plt.subplots(1, 2, figsize=(12,6))

# Неделя 14
axes[0].pie(
    df_14.values,
    labels=df_14.index,
    autopct='%1.1f%%',
    startangle=90,
    explode=explode_14,
    colors=colors_14
)
axes[0].set_title("Неделя 14")

# Недели 19-22
axes[1].pie(
    df_19_22.values,
    labels=df_19_22.index,
    autopct='%1.1f%%',
    startangle=90,
    explode=explode_19_22,
    colors=colors_19_22
)
axes[1].set_title("Недели 19-22")

fig.text(0.5, -0.02, "В некоторых неделях есть аномалия — сервисный сбор 69", 
         ha='center', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# настраиваем фигуру с 1 строкой, 2 столбцами
fig, axes = plt.subplots(1, 2, figsize=(16,6), sharey=True)

# ======= 1. До удаления выбросов =======
sns.violinplot(data=df, x='week', y='service_fee_amt', inner=None, ax=axes[0])
axes[0].axhline(y=69, color='red', linestyle='--', linewidth=1.5, label='Выбросы 69')
axes[0].set_title("До удаления аномалии")
axes[0].set_xlabel("Неделя")
axes[0].set_ylabel("service_fee_amt")

# ======= 2. После удаления выбросов =======
# определяем bad_parties
bad_parties = df[((df['week'] == 14) | (df['week'] >= 19)) & (df['service_fee_amt'] == 69)]['party_rk']
df_clean = df[~df['party_rk'].isin(bad_parties)]

sns.violinplot(data=df_clean, x='week', y='service_fee_amt', inner=None, ax=axes[1])
axes[1].axhline(y=69, color='red', linestyle='--', linewidth=1.5, label='Сбор 69')
axes[1].set_title("После удаления аномалии")
axes[1].set_xlabel("Неделя")
axes[1].set_ylabel("")

# отображаем легенду только один раз
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
fee_order = sorted(df['service_fee_amt'].unique())
df_plot = df.pivot_table(index='week', columns='service_fee_amt', aggfunc='size', fill_value=0)
df_plot = df_plot[fee_order]

# # запасной вариант - очень светлая палитра, не на всех мониторах видна разница
# tbank_colors = [
#     '#f8d81c',  # оригинальный T-Bank жёлтый
#     '#f1ce14',  # яркий фирменный жёлтый
#     '#e6c40c',  # добавленный насыщенный жёлтый
#     '#d4b918',  # сочный насыщенный жёлтый
#     '#b89e17',  # более светлый насыщенный жёлтый
#     '#8c741a',  # тёмный оливково-жёлтый
#     '#8c8c8c',  # светлый, но НЕ бледный серый
#     '#6b6b6b',  # средний серый
#     '#4a5a6b',  # более светлый, но читаемый холодный серый
#     '#2c3844',  # тёмный сине-серый
#     '#1a1a1a'   # почти черный
# ]

tbank_colors = [
    '#ffd600',  # яркий фирменный жёлтый (заметно светлее оригинала)
    '#f3c400',  # насыщенный жёлтый
    '#d4a700',  # тёплый насыщенный тёмно-жёлтый
    '#b08500',  # ещё темнее и контрастнее
    '#8c6a00',  # глубокий жёлто-оливковый
    '#a0a0a0',  # светлый нейтральный серый
    '#787878',  # средний нейтрален, но темнее
    '#505a65',  # холодный тёмно-серый (контрастный с предыдущими)
    '#2f3a45',  # глубокий сине-тёмный
    '#1d1d1d',  # почти чёрный (контраст с #2f3a45)
    '#000000'   # чистый чёрный (ещё один шаг вниз)
]

df_plot.plot(kind='bar', stacked=True, figsize=(12, 6), color=tbank_colors)

plt.xlabel('Номер недели')
plt.ylabel('Количество записей')
plt.title('Распределение сервисного сбора по неделям')
plt.legend(title='Сервисный Сбор', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.xticks(rotation=0)
plt.show()

In [ ]:
colors = [
    '#ffd600',  # яркий фирменный жёлтый (заметно светлее оригинала)
    # '#f3c400',  # насыщенный жёлтый
    # '#d4a700',  # тёплый насыщенный тёмно-жёлтый
    # '#b08500',  # ещё темнее и контрастнее
    # '#8c6a00',  # глубокий жёлто-оливковый
    '#a0a0a0',  # светлый нейтральный серый
    # '#787878',  # средний нейтрален, но темнее
    # '#505a65',  # холодный тёмно-серый (контрастный с предыдущими)
    # '#2f3a45',  # глубокий сине-тёмный
    # '#1d1d1d',  # почти чёрный (контраст с #2f3a45)
    '#000000'   # чистый чёрный (ещё один шаг вниз)
]

fee_order = df[df['week'] == 14].groupby('service_fee_amt')['party_rk'].nunique()
fee_order.plot(kind='bar', stacked=True, figsize=(4,6), color=colors)

# plt.legend().remove()
plt.xlabel('')
plt.ylabel('Количество записей')
plt.title('Сервисный сбор в неделю 14')
# plt.legend(title='Сервисный Сбор', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.legend(title='Сервисный Сбор', loc='upper left')
plt.tight_layout()
plt.xticks(rotation=0)
plt.show()

In [ ]:
colors = [
    '#ffd600',  # яркий фирменный жёлтый (заметно светлее оригинала)
    '#f3c400',  # насыщенный жёлтый
    '#d4a700',  # тёплый насыщенный тёмно-жёлтый
    '#b08500',  # ещё темнее и контрастнее
    '#8c6a00',  # глубокий жёлто-оливковый
    '#a0a0a0',  # светлый нейтральный серый
    '#787878',  # средний нейтрален, но темнее
    '#505a65',  # холодный тёмно-серый (контрастный с предыдущими)
    '#2f3a45',  # глубокий сине-тёмный
    '#1d1d1d',  # почти чёрный (контраст с #2f3a45)
    '#000000'   # чистый чёрный (ещё один шаг вниз)
]

fee_order = df[df['week'].isin([15, 16, 17])].groupby('service_fee_amt')['party_rk'].nunique()
fee_order.plot(kind='bar', stacked=True, figsize=(8,6), color=colors)

# plt.legend().remove()
plt.xlabel('')
plt.ylabel('Количество записей')
plt.title('Сервисный сбор в недели 15-17')
# plt.legend(title='Сервисный Сбор', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.legend(title='Сервисный Сбор', loc='upper left')
plt.tight_layout()
plt.xticks(rotation=0)
plt.show()

In [ ]:
colors = [
    '#ffd600',  # яркий фирменный жёлтый (заметно светлее оригинала)
    '#f3c400',  # насыщенный жёлтый
    '#d4a700',  # тёплый насыщенный тёмно-жёлтый
    '#b08500',  # ещё темнее и контрастнее
    '#8c6a00',  # глубокий жёлто-оливковый
    '#a0a0a0',  # светлый нейтральный серый
    '#787878',  # средний нейтрален, но темнее
    '#505a65',  # холодный тёмно-серый (контрастный с предыдущими)
    '#2f3a45',  # глубокий сине-тёмный
    '#1d1d1d',  # почти чёрный (контраст с #2f3a45)
    '#000000'   # чистый чёрный (ещё один шаг вниз)
]

fee_order = df[df['week'].isin([19, 20, 21, 22])].groupby('service_fee_amt')['party_rk'].nunique()
fee_order.plot(kind='bar', stacked=True, figsize=(5,6), color=colors)

# plt.legend().remove()
plt.xlabel('')
plt.ylabel('Количество записей')
plt.title('Сервисный сбор в недели 19-22')
# plt.legend(title='Сервисный Сбор', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.legend(title='Сервисный Сбор', loc='upper left')
plt.tight_layout()
plt.xticks(rotation=0)
plt.show()

In [ ]:
colors = [
    '#ffd600',  # яркий фирменный жёлтый (заметно светлее оригинала)
    '#f3c400',  # насыщенный жёлтый
    '#d4a700',  # тёплый насыщенный тёмно-жёлтый
    '#b08500',  # ещё темнее и контрастнее
    '#8c6a00',  # глубокий жёлто-оливковый
    '#a0a0a0',  # светлый нейтральный серый
    '#787878',  # средний нейтрален, но темнее
    '#505a65',  # холодный тёмно-серый (контрастный с предыдущими)
    '#2f3a45',  # глубокий сине-тёмный
    '#1d1d1d',  # почти чёрный (контраст с #2f3a45)
    '#000000'   # чистый чёрный (ещё один шаг вниз)
]

fee_order = df[df['week'] == 18].groupby('service_fee_amt')['party_rk'].nunique()
fee_order.plot(kind='bar', stacked=True, figsize=(10,2), color=colors)

# plt.legend().remove()
plt.xlabel('')
plt.ylabel('Количество записей')
plt.title('Сервисный сбор в неделю 18')
# plt.legend(title='Сервисный Сбор', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.legend(title='Сервисный Сбор', loc='upper left')
plt.tight_layout()
plt.xticks(rotation=0)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Преобразуем first_order_date в datetime, если ещё не
df['first_order_date'] = pd.to_datetime(df['first_order_date'])

# Считаем количество заказов по неделям
orders_per_week = df.groupby(df['first_order_date'].dt.to_period('W')).size().sort_index()
orders_per_week.index = orders_per_week.index.to_timestamp()  # для корректного отображения на оси X

# Фирменные цвета
tbank_colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00', 
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]

# Выбираем основной желтый цвет
main_color = tbank_colors[0]

# Стильный график
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(orders_per_week.index, orders_per_week.values, color=main_color, width=5)  # width=5 дней, можно менять

# Подписи
ax.set_xlabel('Дата первого заказа')
ax.set_ylabel('Количество заказов')
ax.set_title('Распределение заказов по дате первого заказа', fontsize=14, fontweight='bold')

# Сетка и стиль
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Задаём диапазон дат
start_date = '2025-04-28'
end_date   = '2025-05-04'

# Фильтруем данные по неделе 18 и диапазону first_order_date
df_week18 = df[
    (df['week'] == 18) &
    (df['first_order_date'] >= start_date) &
    (df['first_order_date'] <= end_date)
].copy()

# Преобразуем first_order_date в datetime, если нужно
df_week18['first_order_date'] = pd.to_datetime(df_week18['first_order_date']).dt.date

# Считаем количество записей по дате и по service_fee_amt
df_plot = df_week18.groupby(['first_order_date', 'service_fee_amt']).size().unstack(fill_value=0)

# Фирменные цвета (возьмем столько, сколько столбцов)
tbank_colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00', 
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]
colors = tbank_colors[:len(df_plot.columns)]

# Построение stacked bar plot
fig, ax = plt.subplots(figsize=(12, 4))
df_plot.plot(kind='bar', stacked=True, color=colors, ax=ax)

# Подписи и стиль
ax.set_xlabel('Дата первого заказа')
ax.set_ylabel('Количество записей')
ax.set_title('Распределение сервисного сбора по first_order_date (week=18)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
ax.legend(title='Service Fee', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_plot = df.copy()
df_plot = df_plot.drop(['week_start', 'week_end'], axis=1)

plt.figure(figsize=(14, 3))
sns.heatmap(df_plot.isnull(), cbar=False, cmap="viridis")
plt.title("Карта пропусков в датасете")
plt.show()

In [ ]:
df_plot = df.copy()
df_plot = df_plot.drop(['week_start', 'week_end'], axis=1)

missing = df_plot.isnull().mean().sort_values(ascending=False)

plt.figure(figsize=(10, 3))
missing.plot(kind='bar', color='#ffd600')
plt.title('Доля пропусков по колонкам')
plt.ylabel('Доля')
plt.show()


In [ ]:
df.describe().T


In [ ]:
num_cols = ['gmv', 'orders_cnt', 'liters', 'entries_cnt', 'age', 'children_cnt']
for col in num_cols:
    plt.figure(figsize=(8, 3))
    sns.boxenplot(x=df[col], color="#ffd600")
    plt.title(f"Распределение {col}")
    plt.show()


In [ ]:
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True, color="#505a65")
    plt.title(f"Распределение {col}")
    plt.show()


In [ ]:
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col], color='#ffd600')
    sns.stripplot(x=df[col], color="red", size=2, alpha=0.5)
    plt.title(f"Выбросы в {col}")
    plt.show()


In [ ]:
# df.describe().T

num_cols = ['service_fee_amt', 'cashback_category', 'gmv', 'orders_cnt', 'liters', 'entries_cnt', 'age', 'children_cnt']

plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="YlGnBu")
plt.title("Корреляционная матрица")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import numpy as np

# фирменные цвета
tbank_colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]

tbank_cmap = mcolors.LinearSegmentedColormap.from_list("tbank", tbank_colors)

num_cols = ['service_fee_amt', 'cashback_category', 'gmv',
            'orders_cnt', 'liters', 'entries_cnt', 'age', 'children_cnt']

corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))

# Основная heatmap без текста
ax = sns.heatmap(
    corr,
    cmap=tbank_cmap,
    linewidths=0.5,
    linecolor="#e0e0e0",
    cbar=True,
    annot=False   # <-- выключаем встроенные подписи
)

# Добавляем подписи вручную
# Получаем нормализатор под colormap
norm = mcolors.Normalize(vmin=corr.min().min(), vmax=corr.max().max())

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        val = corr.iloc[i, j]
        
        # цвет фона ячейки
        rgba = tbank_cmap(norm(val))
        r, g, b, _ = rgba
        
        # определяем яркость (0=черный, 1=белый)
        luminance = 0.299*r + 0.587*g + 0.114*b
        
        # если фон светлый → текст чёрный, если тёмный → белый
        text_color = "black" if luminance > 0.6 else "white"
        
        ax.text(
            j + 0.5, i + 0.5,
            f"{val:.2f}",
            color=text_color,
            ha='center', va='center', fontsize=10
        )

plt.title("Корреляционная матрица", 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(x=df['orders_cnt'], color="#ffd600")
plt.title("Количество клиентов по orders_cnt")
plt.show()


In [ ]:
mins = df[num_cols].min().reset_index()
mins.columns = ["feature", "min"]

plt.figure(figsize=(6, 3))
sns.barplot(data=mins, x="feature", y="min", palette="viridis")
plt.xticks(rotation=45)
plt.title("Минимальные значения по признакам")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Фирменные цвета Т‑банка
colors = [
    '#ffd600',  # яркий
    '#f3c400',  # насыщенный
    '#d4a700',  # тёплый насыщенный тёмно-жёлтый
    '#b08500',  # ещё темнее
    '#8c6a00',  # глубокий жёлто-оливковый
]

# Уровни сервисного сбора
fee_levels = np.array([0, 19, 23, 29, 39])
groups = [(0,), (19,23), (29,39)]

# Симулируем гипотетический отток: плавно внутри групп
churn_rates = []
for group in groups:
    base = 0.05 + 0.05*len(churn_rates)  # базовый уровень
    for i, fee in enumerate(group):
        rate = base + 0.02*i  # гипотетический плавный рост
        churn_rates.append(rate)

df_plot = pd.DataFrame({
    'service_fee_amt': fee_levels,
    'churn_rate': churn_rates
})

plt.figure(figsize=(8,5))

# Линия и точки (с фирменным жёлтым)
sns.lineplot(
    data=df_plot, 
    x='service_fee_amt', 
    y='churn_rate', 
    marker='o', 
    markersize=10, 
    linewidth=2,
    color=colors[2],  # #d4a700
    sort=False
)

# Красная линия разрыва между группами 23 и 29
plt.vlines(x=26, ymin=0, ymax=0.25, color='red', linestyle='--', linewidth=2, label='Гипотетический разрыв')

plt.title("Гипотетическая зависимость вероятности оттока от сервисного сбора\nпри подтверждении гипотезы", fontsize=14)
plt.xlabel("Сервисный сбор, руб.", fontsize=12)
plt.ylabel("Вероятность оттока", fontsize=12)
plt.xticks(fee_levels)
plt.ylim(0, 0.25)
plt.grid(True, alpha=0.3)
plt.legend()
plt.gca().set_yticklabels([])

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- гипотетические данные ---
service_fees = np.array([0, 19, 23, 29, 39])
# плавный рост внутри групп, без конкретных чисел по Y
y_values = np.array([0.2, 0.25, 0.27, 0.37, 0.4])  

# фирменные цвета Т‑Банка
colors = [
    '#ffd600',  # яркий жёлтый
    '#f3c400',
    '#d4a700',
    '#b08500',
    '#8c6a00',
]

plt.figure(figsize=(8,5))

# рисуем линии между точками с выделением разрыва красным
line_colors = colors.copy()
line_colors[2] = 'red'  # линия между 23 и 29

for i in range(len(service_fees)-1):
    plt.plot(service_fees[i:i+2], y_values[i:i+2], color=line_colors[i], linewidth=3)

# точки
plt.scatter(service_fees, y_values, color=colors, s=100, zorder=5)

# оформление
plt.title('Гипотетическая зависимость оттока от сервисного сбора\n(при подтверждении H1)', fontsize=14)
plt.xlabel('Сервисный сбор', fontsize=12)
plt.xticks(service_fees)
plt.yticks([])  # убираем значения по оси Y
plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Данные таблицы
data = {
    "Сбор": ["0", "19", "23", "29", "39"],
    "Без контроля": ["Баз.", "-0.1944", "-0.2748", "-0.3340", "-0.7117"],
    "С контролем": ["Баз.", "+0.0996", "+0.0334", "-0.0575", "-0.3619"],
    "Изменение": ["—", "↗ положительный", "↔ незначим", "↔ незначим", "↘ остался отрицательным"]
}

df = pd.DataFrame(data)

# Фирменные цвета Т-Банка
colors = [
    '#ffd600',  # яркий желтый
    '#f3c400',  # насыщенный желтый
    '#d4a700',  # теплый темно-желтый
    '#b08500',  # еще темнее
    '#8c6a00',  # глубокий желто-оливковый
    '#a0a0a0',  # светлый серый
    '#787878',  # средний серый
    '#505a65',  # темный серый
    '#2f3a45',  # глубокий сине-темный
    '#1d1d1d',  # почти черный
    '#000000'   # чистый черный
]

# Создаем фигуру
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')  # убираем оси

col_widths = [0.15, 0.25, 0.25, 0.35]  # сумма = 1
# Цвета строк (градиент по желтому)
row_colors = [colors[1], colors[2], colors[2], colors[2], colors[3]]

# Создаем таблицу без лишней первой колонки
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 cellLoc='center',
                 loc='center',
                 colColours=[colors[4]]*4,
                 rowColours=row_colors,
                 colWidths=col_widths)

# Настройка стиля
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)

# Дополнительное выделение: жирный шрифт для значимых коэффициентов
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')  # заголовки белым
    elif j in [1, 2] and '***' in str(cell.get_text().get_text()):
        cell.set_text_props(weight='bold')  # значимые коэффициенты

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Данные
data = {
    "Сбор": ["19", "23", "29", "39"],
    "Коэффициент": [-0.1944, -0.2748, -0.3340, -0.7117],
    "P-value": [0.001, 0.001, 0.001, 0.001]
}

# Рассчитаем OR и изменение шансов
df = pd.DataFrame(data)
df["OR"] = np.exp(df["Коэффициент"]).round(2)
df["Δ шансов"] = ((df["OR"] - 1)*100).round(0)
df["Δ шансов"] = df["Δ шансов"].apply(lambda x: f"↑{abs(x)}%" if x>0 else f"↓{abs(x)}%")

# Фирменные цвета Т-Банка
colors = ['#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
          '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000']

row_colors = [colors[2], colors[2], colors[3], colors[3]]  # оттенки желтого для строк

# Создаем фигуру
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

# Ширины колонок
col_widths = [0.15, 0.25, 0.15, 0.2, 0.25]

# Создаем таблицу
table = ax.table(cellText=df[["Сбор", "Коэффициент", "P-value", "OR", "Δ шансов"]].values,
                 colLabels=["Сбор", "Коэффициент", "P-value", "OR", "Δ шансов"],
                 cellLoc='center',
                 loc='center',
                 colColours=[colors[4]]*5,
                 rowColours=row_colors,
                 colWidths=col_widths)

# Настройка шрифта
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)

# Выделяем заголовки и значимые коэффициенты
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
    elif j in [1, 2] and df.iloc[i-1, j] != 0:
        cell.set_text_props(weight='bold')
    elif j == 4:  # Δ шансов: красный ↓, зеленый ↑
        text = cell.get_text().get_text()
        if "↓" in text:
            cell.get_text().set_color('red')
        elif "↑" in text:
            cell.get_text().set_color('green')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Данные
data = {
    "Уровень сбора": ["0 (константа)", "19", "23", "29", "39"],
    "Коэффициент": [-0.3090, 0.1145, 0.0341, -0.0251, -0.4027],
    "P-value": [0.001, 0.001, 0.064, 0.434, 0.001],
    "Интерпретация": [
        "Базовый уровень",
        "Увеличивает шансы возврата",
        "Незначимое влияние",
        "Незначимое влияние",
        "Сильно снижает шансы возврата"
    ]
}

df = pd.DataFrame(data)

# Фирменные цвета Т-Банка
colors = ['#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
          '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000']

# Создаем фигуру
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

# Ширины колонок (интерпретация шире)
col_widths = [0.25, 0.2, 0.1, 0.45]

# Создаем таблицу
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 cellLoc='center',
                 loc='center',
                 colColours=[colors[4]]*4,
                 colWidths=col_widths,
                 rowLabels=None)

# Настройка шрифта
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)

# Выделяем заголовки и значимые коэффициенты
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
    elif i > 0:
        # Жирный для значимых коэффициентов (p<0.05)
        if df.iloc[i-1, 2] < 0.05:
            cell.set_text_props(weight='bold')
            # Зеленый для положительного, красный для отрицательного
            if j == 1:
                if df.iloc[i-1, j] > 0:
                    cell.get_text().set_color('green')
                else:
                    cell.get_text().set_color('red')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Данные
data = {
    "Сбор": ["0 (константа)", "19", "23", "29", "39"],
    "Коэффициент": [-0.3090, 0.1145, 0.0341, -0.0251, -0.4027],
    "P-value": [0.001, 0.001, 0.064, 0.434, 0.001],
    "Интерпретация": [
        "Базовый уровень",
        "Увеличивает шансы возврата",
        "Незначимое влияние",
        "Незначимое влияние",
        "Сильно снижает шансы возврата"
    ]
}

df = pd.DataFrame(data)

# Рассчитаем OR и Δ шансов
df["OR"] = np.exp(df["Коэффициент"]).round(2)
df["Δ шансов"] = ((df["OR"] - 1)*100).round(0)
df["Δ шансов"] = df["Δ шансов"].apply(lambda x: f"↑{abs(x)}%" if x>0 else f"↓{abs(x)}%")

# Для базового уровня ставим прочерки
df.loc[0, ["OR", "Δ шансов"]] = "-"

# Цвета Т-Банка
colors = ['#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
          '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000']

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

col_widths = [0.15, 0.15, 0.1, 0.1, 0.15, 0.35]  # расширяем последнюю колонку при необходимости

table = ax.table(
    cellText=df[["Сбор", "Коэффициент", "P-value", "OR", "Δ шансов", "Интерпретация"]].values,
    colLabels=["Сбор", "Коэффициент", "P-value", "OR*", "Δ шансов", "Интерпретация"],
    cellLoc='center',
    loc='center',
    colColours=[colors[4]]*6,
    colWidths=col_widths
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)

# Выделяем заголовки и значимые коэффициенты
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
    elif i > 0:
        # Жирный для значимых коэффициентов (p<0.05)
        if df.iloc[i-1, 2] < 0.05:
            cell.set_text_props(weight='bold')
            # Цвет коэффициента: зеленый если положительный, красный если отрицательный
            if j == 1:
                if df.iloc[i-1, j] > 0:
                    cell.get_text().set_color('green')
                else:
                    cell.get_text().set_color('red')
        # Цвет Δ шансов: красный ↓, зеленый ↑
        if j == 4:
            text = cell.get_text().get_text()
            if "↓" in text:
                cell.get_text().set_color('red')
            elif "↑" in text:
                cell.get_text().set_color('green')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Данные модели с контролями + кэшбек
data_full = {
    "Переменная": [
        "Intercept", "19", "23", "29", "39",
        "platform[T.IOS]", "platform[T.WEB]", "platform[T.WEBVIEW_ANDROID]",
        "platform[T.WEBVIEW_IOS]", "cashback_category", "np.log1p(gmv)", "orders_cnt", "entries_cnt"
    ],
    "Коэффициент": [
        -0.1620, 0.0907, 0.0320, -0.0469, -0.3496,
        0.0521, 0.1137, -0.0175, 0.1402, 0.0206, -0.0934, 0.2201, 0.0651
    ],
    "P-value": [
        0.007, 0.000, 0.087, 0.149, 0.000,
        0.000, 0.001, 0.412, 0.000, 0.000, 0.000, 0.000, 0.000
    ],
    "Интерпретация": [
        "Базовый уровень", "Увеличивает шансы возврата", "Незначимое влияние",
        "Незначимое влияние", "Сильно снижает шансы возврата",
        "IOS vs базовая", "WEB vs базовая", "WEBVIEW_ANDROID vs базовая",
        "WEBVIEW_IOS vs базовая", "На 1% кэшбека", "На 1 единицу GMV", "На 1 заказ", "На 1 вход"
    ]
}

df_full = pd.DataFrame(data_full)

# OR и Δ шансов
df_full["OR"] = np.exp(df_full["Коэффициент"]).round(2)
df_full["Δ шансов"] = ((df_full["OR"] - 1)*100).round(0)
df_full["Δ шансов"] = df_full["Δ шансов"].apply(lambda x: f"↑{abs(x)}%" if x>0 else f"↓{abs(x)}%")
# Прочерки для базового уровня
df_full.loc[0, ["OR", "Δ шансов"]] = "-"

# Цвета Т-Банка
colors = ['#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
          '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000']

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

col_widths = [0.26, 0.15, 0.09, 0.09, 0.09, 0.32]

table = ax.table(
    cellText=df_full[["Переменная", "Коэффициент", "P-value", "OR", "Δ шансов", "Интерпретация"]].values,
    colLabels=["Переменная", "Коэффициент", "P-value", "OR*", "Δ шансов", "Интерпретация"],
    cellLoc='center',
    loc='center',
    colColours=[colors[4]]*6,
    colWidths=col_widths
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.2)

# Выделяем заголовки и значимые коэффициенты
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
    elif i > 0:
        # Жирный для значимых коэффициентов (p<0.05)
        if df_full.iloc[i-1, 2] < 0.05:
            cell.set_text_props(weight='bold')
            if j == 1 and df_full.iloc[i-1, j] != 0:
                if df_full.iloc[i-1, j] > 0:
                    cell.get_text().set_color('green')
                else:
                    cell.get_text().set_color('red')
        # Цвет Δ шансов: красный ↓, зеленый ↑
        if j == 4 and df_full.iloc[i-1, j] != "-":
            text = cell.get_text().get_text()
            if "↓" in text:
                cell.get_text().set_color('red')
            elif "↑" in text:
                cell.get_text().set_color('green')

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Данные из твоей новой модели
data_new = {
    "Переменная": [
        "Intercept", "fee_cat_19", "fee_cat_23", "fee_cat_29", "fee_cat_39",
        "cashback_bin_[9-34]", "gmv_bin_5000-15000", "gmv_bin_15000+",
        "orders_bin_4+", "entries_bin_5-10", "entries_bin_10+", "entries_bin_missing",
        "platform_IOS", "platform_WEB", "platform_WEBVIEW_ANDROID", "platform_WEBVIEW_IOS"
    ],
    "Коэффициент": [
        -0.3983, 0.0925, 0.0315, -0.0100, -0.3727,
        0.4254, 0.1954, 0.0382,
        0.2475, 0.4657, 0.7459, 0.1411,
        0.0472, 0.0682, -0.0485, 0.0841
    ],
    "P-value": [
        0.000, 0.000, 0.070, 0.739, 0.000,
        0.000, 0.000, 0.720,
        0.000, 0.000, 0.000, 0.000,
        0.000, 0.027, 0.009, 0.000
    ],
    "Интерпретация": [
        "Базовый уровень", "↑ шансы", "Незначимо", "Незначимо", "↓ шансы",
        "↑ шансы (кэшбек)", "↑ шансы (GMV 5-15k)", "Незначимо (GMV>15k)",
        "↑ шансы (orders>4)", "↑ шансы (entries 5-10)", "↑ шансы (entries>10)", "↑ шансы (missing entries)",
        "IOS vs базовая", "WEB vs базовая", "WEBVIEW_ANDROID vs базовая", "WEBVIEW_IOS vs базовая"
    ]
}

df_new = pd.DataFrame(data_new)

# Рассчитаем OR и Δ шансы
df_new["OR"] = np.exp(df_new["Коэффициент"]).round(2)
df_new["Δ шансов"] = ((df_new["OR"] - 1) * 100).round(0)
df_new["Δ шансов"] = df_new["Δ шансов"].apply(lambda x: f"↑{abs(x)}%" if x > 0 else f"↓{abs(x)}%")
df_new.loc[0, ["OR", "Δ шансов"]] = "-"

# Цвета Т-Банка
colors = ['#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
          '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000']

# Рисуем таблицу
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')

col_widths = [0.25, 0.12, 0.08, 0.08, 0.12, 0.35]  # ширины колонок

table = ax.table(
    cellText=df_new[["Переменная", "Коэффициент", "P-value", "OR", "Δ шансов", "Интерпретация"]].values,
    colLabels=["Переменная", "Коэффициент", "P-value", "OR*", "Δ шансов", "Интерпретация"],
    cellLoc='center',
    loc='center',
    colColours=[colors[4]]*6,
    colWidths=col_widths
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.2)

# Выделяем заголовки и значимые коэффициенты
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
    elif i > 0:
        # Жирный для значимых коэффициентов (p<0.05)
        if df_new.iloc[i-1, 2] < 0.05:
            cell.set_text_props(weight='bold')
            if j == 1:
                if df_new.iloc[i-1, j] > 0:
                    cell.get_text().set_color('green')
                elif df_new.iloc[i-1, j] < 0:
                    cell.get_text().set_color('red')
        # Цвет Δ шансов: красный ↓, зеленый ↑
        if j == 4 and df_new.iloc[i-1, j] != "-":
            text = cell.get_text().get_text()
            if "↓" in text:
                cell.get_text().set_color('red')
            elif "↑" in text:
                cell.get_text().set_color('green')

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.table import Table

# Данные
data = {
    "Значение сбора": ["29", "39"],
    "Retention N+1": ["41.7%", "32.9%"],
    "Разница": ["—", "-8.8%"],
    "Z-статистика": ["—", "8.96"],
    "P-value": ["—", "<0.001"]
}

df = pd.DataFrame(data)

# Фирменные цвета Т‑Банк
colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]

# Создаем фигуру
fig, ax = plt.subplots(figsize=(8, 2))
ax.axis('off')

# Создаем таблицу
table = Table(ax, bbox=[0, 0, 1, 1])

n_rows, n_cols = df.shape

# Размер ячеек
width = 1.0 / n_cols
height = 1.0 / (n_rows + 1)

# Заголовки
for i, col in enumerate(df.columns):
    cell = table.add_cell(0, i, width, height, text=col, loc='center', facecolor=colors[1])
    cell.get_text().set_weight('bold')
    cell.get_text().set_color('black')

# Данные
for row in range(n_rows):
    for col in range(n_cols):
        val = df.iloc[row, col]
        # Выделяем отрицательную разницу красным
        if val == "-8.8%":
            cell_color = colors[0]
        else:
            cell_color = "#ffffff"
        cell = table.add_cell(row+1, col, width, height, text=val, loc='center', facecolor=cell_color)
        cell.get_text().set_color('black')

ax.add_table(table)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.table import Table
import numpy as np

# Данные VIF
data = {
    "const": 4.089928,
    "fee_cat_19": 1.183018,
    "fee_cat_23": 1.108618,
    "fee_cat_29": 1.036479,
    "fee_cat_39": 1.047158,
    "cashback_bin_(9.0, 34.0]": 1.012369,
    "gmv_bin_15000+": 1.061114,
    "gmv_bin_5000-15000": 1.140939,
    "orders_bin_(4.0, 19.0]": 1.375639,
    "entries_bin_10+": 1.221102,
    "entries_bin_5-10": 1.168302,
    "entries_bin_missing": 1.002079,
    "platform_IOS": 1.146795,
    "platform_WEB": 1.024306,
    "platform_WEBVIEW_ANDROID": 1.059266,
    "platform_WEBVIEW_IOS": 1.155377
}

columns = list(data.keys())
values = list(data.values())

# Разделяем на две строки
half = int(np.ceil(len(columns)/2))
cols_top, cols_bottom = columns[:half], columns[half:]
vals_top, vals_bottom = values[:half], values[half:]

fig, ax = plt.subplots(figsize=(20, 4))
ax.axis('off')

table = Table(ax, bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)

def add_row(table, row_idx, cols, vals, header_color='#f3c400', cell_color='#ffffff'):
    n_cols = len(cols)
    width = 1.0 / n_cols
    height_header = 0.3
    height_cell = 0.7
    # Заголовки
    for i, col in enumerate(cols):
        cell = table.add_cell(row_idx, i, width, height_header, text=col, loc='center', facecolor=header_color)
        cell.get_text().set_weight('bold')
        cell.get_text().set_fontsize(8)
    # Значения
    for i, val in enumerate(vals):
        cell = table.add_cell(row_idx+1, i, width, height_cell, text=f"{val:.3f}", loc='center', facecolor=cell_color)
        cell.get_text().set_fontsize(30)

# Верхняя строка
add_row(table, 0, cols_top, vals_top)
# Нижняя строка
add_row(table, 2, cols_bottom, vals_bottom)

ax.add_table(table)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.DataFrame({
    "week": ["14", "15", "16", "17", "18", "19"],
    "order_cnt": [3, 0, 5, 0, 2, 1],
    "entries_cnt": [12, None, 17, None, 9, 13]
})

x = range(len(data))

plt.figure(figsize=(12, 6))

# Линии - чёрно-серая и жёлтая гамма
plt.plot(x, data.order_cnt, marker="o", label="order_cnt", color='#333333', linewidth=2)  # Тёмно-серый
plt.plot(x, data.entries_cnt, marker="o", label="entries_cnt", 
         color='#FFD100', linewidth=2,  # Жёлтый Т-Банка
         markersize=12,  # Увеличен размер точек
         markeredgecolor='#000000',  # Чёрная обводка для контраста
         markeredgewidth=1.5,  # Толщина обводки
         markerfacecolor='#FFD100')  # Заливка жёлтым

# Подсветим зоны, где entries_cnt отсутствует - КРАСНЫМ
for i in x:
    if pd.isna(data.entries_cnt.iloc[i]):
        plt.axvspan(i-0.3, i+0.3, color='red', alpha=0.3)  # Красная подсветка

plt.xticks(x, data.week, rotation=45)
plt.title("Отсутствующие значения entries_cnt в недели без заказов (иллюстрация для одного клиента)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Данные
data = {
    "Клиент": [1, 1, 1, 2, 2, 2],
    "Неделя": [1, 2, 3, 1, 2, 3],
    "orders_cnt": [2, 0, 1, 1, 2, 0],
    "entries_cnt": [4, 2, 2, 1, 4, 2]
}

df = pd.DataFrame(data)

# Цвета для таблицы
colors = ['#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00']

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

col_widths = [0.1, 0.1, 0.15, 0.15]

table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc='center',
    loc='center',
    colColours=[colors[4]]*4,
    colWidths=col_widths
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)

# Выделяем заголовки
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')

# Красный фон для проблемных строк (номера 2 и 6 без заголовка → i=2 и i=6)
problem_rows = [2, 6]
for i in problem_rows:
    for j in range(4):
        table[i, j].set_facecolor('red')
        table[i, j].get_text().set_color('white')

plt.show()


In [ ]:
# Чистка датасета

import pandas as pd
df = pd.read_csv("dataset_for_dano_fuel.csv")

df["week"] = pd.to_datetime(df["week"])

weeks_sorted = sorted(df["week"].unique())
print(weeks_sorted)

df["week"] = df["week"].dt.isocalendar().week
df["week"] = df["week"] + 1

weeks_sorted = sorted(df["week"].unique())

first_week = weeks_sorted[0]
last_4_weeks = weeks_sorted[-4:]

outlier_mask = (df["service_fee_amt"] == 69) & (
    (df["week"] == first_week) | (df["week"].isin(last_4_weeks))
)

print(f"Удаляем выбросов: {outlier_mask.sum()} строк")
df = df.loc[~outlier_mask].copy()

df.drop(columns=["region"], inplace=True)

print("Итоговые недели:", sorted(df["week"].unique()))

df.to_csv("dataset_for_dano_fuel_clean.csv", index=False)